# 1b Nucleoid Mapping to Cells

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import skimage as sk
from sklearn import mixture
import pandas as pd

import dask.dataframe as dd
import matplotlib.patches as patches
from dask.distributed import wait

import xarray as xr
import matplotlib.colors as mcolors

colors = list(mcolors.TABLEAU_COLORS.values())
%matplotlib inline

# import paulssonlab.paulssonlab.src.paulssonlab.deaton.trenchripper.trenchripper as tr

import paulssonlab.deaton.trenchripper.trenchripper as tr

import warnings
warnings.filterwarnings(action="once")

import matplotlib
matplotlib.rcParams["figure.figsize"] = [20, 10]

In [ ]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="1:00:00",
    local=False,
    n_workers=20,
    n_workers_min=20,
    memory="8GB", #5GB is good. 2GB after
    working_directory="/home/lag36/scratch/lag36/dask",
)
dask_controller.startdask()
dask_controller.displaydashboard()

In [ ]:
def assign_nucleoids_to_cells_df(nuc_df, cell_df):
    
    nuc_df_output = nuc_df
    nuc_df = nuc_df.reset_index().loc[:, ('File Trench Index', 'timepoints','Objectid', 'centroid_x', 'centroid_y', 'File Parquet Index')].set_index(['File Trench Index','timepoints', 'Objectid'])
    cell_df = cell_df.reset_index().loc[:,('File Trench Index', 'timepoints', 'Segment Index' , 'bbox_min_row', 'bbox_min_col', 'bbox_max_row', 'bbox_max_col', 'File Parquet Index')].set_index(['File Trench Index', "timepoints", "Segment Index"])

    nuc_array = nuc_df.to_xarray().to_array().to_numpy()
    nuc_array = np.moveaxis(nuc_array, 0, -1)
    
    cell_array = cell_df.to_xarray().to_array().to_numpy()
    cell_array = np.moveaxis(cell_array, 0, -1)

    del nuc_df, cell_df
    
    nuc_df_output["Segment Index"], nuc_df_output["Cells Per Nucleoid"], nuc_df_output["Nuc Parquet Index"], nuc_df_output["Cell Parquet Index"] = assign_nucleoids_to_cells(nuc_array = nuc_array, cell_array = cell_array)
    
    return nuc_df_output
    
#     mapping, cells_per_nuc, nuc_parquet, cells_parquet = assign_nucleoids_to_cells(nuc_array = nuc_array, cell_array = cell_array)
    
#     del nuc_array, cell_array
#     nuc_df_output["Segment Index"] = mapping
#     nuc_df_output["Cells Per Nucleoid"] = cells_per_nuc
#     nuc_df_output["Nuc Parquet Index"] = nuc_parquet
#     nuc_df_output["Cell Parquet Index"] = cells_parquet
#     return nuc_df_output

def assign_nucleoids_to_cells(nuc_array, cell_array):
    """
    nuc_array and cell_array are numpy arrays of dimensions:
    (# Trench ID indices x # Timepoints x Max number of cells per timepoint x Max number of nucleoids per timepoint)
    """
    
    # These ones have dimensions trench_idx X timepoints X  Max # nucleoids per image X 1
    x = nuc_array[:,:,:,0][:,:,:,None] # x coordinate of nucleoid centroid
    y = nuc_array[:,:,:,1][:,:,:,None] # y coordiate of nucleoid centroid

    # These ones have dimensions trench_idx X timepoints X  1 X Max # of cells per image
    min_row = cell_array[:,:,:,0][:,:,None,:]
    min_col = cell_array[:,:,:,1][:,:,None,:]
    max_row = cell_array[:,:,:,2][:,:,None,:]
    max_col = cell_array[:,:,:,3][:,:,None,:]

    # It compares all nucleoids with all cells
    # Is x greater than min_column?
    x_g_min_col = x > min_col
    # Is x less than max_colum?n
    x_l_max_col = x < max_col
    # Is y greater than min_row?
    y_g_min_row = y > min_row
    y_l_max_row = y < max_row

    
    '''
    Each element is Trench id, timepoint, nucleoid id, cell id of nucleoid - cell pair that was true on the previous array
    It will match the length of nuc_df if there are no multiple dfs.

    - This will have trouble if a nucleoid is mapped 0 or >1 times.
    '''
    inside = x_g_min_col & x_l_max_col & y_g_min_row & y_l_max_row
    
    '''
    2022-10-26
    '''
    # Finds the number of cells per nucleoid
    n_cells_per_nuc_collapsed = np.sum(inside, axis=3)

    # Get nans
    not_nan = ~(np.isnan(x) | np.isnan(min_row))
    not_nan_collapsed = np.any(not_nan,axis=3)

    # Add a column of true to the end of the "inside" array
    inside2 = np.append(inside, 
                        np.ones((inside.shape[0], inside.shape[1], inside.shape[2], 1), dtype=bool), axis=3)
    first_cell = np.argmax(inside2, axis=3)

    # Selects nucleoids that have no cells but are not nan, or have one or more cells
    nuc_has_zero_cells = first_cell == inside.shape[3]
    nuc_has_zero_cells_but_not_nan = (nuc_has_zero_cells) & (not_nan_collapsed)
    nuc_has_one_or_more_cells = n_cells_per_nuc_collapsed > 0

    # ONLY NO MAPPED NUCLEOIDS: Modify first cell to include -1 if the nucleoid was not mapped to any real cell
    # This will happen if the nucleoid is mapped to cell inside.shape[3]
    first_cell[nuc_has_zero_cells] = -1

    # Boolean array (trench id X timepoint X nucleoid ): True if nucleoid is selected as valid
    nuc_selected = nuc_has_zero_cells_but_not_nan | nuc_has_one_or_more_cells

    # Recover parquet index of selected nucleoids and cells
    nuc_selected_idx = np.nonzero(nuc_selected)

    '''
    GENERATE THE LISTS
    '''
    # Nucleoid parquet indices
    nuc_parquet_ids = nuc_array[nuc_selected_idx[0:3]][:,2].astype('int64')

    # Cell parquet indices
    # It will be nan if nucleoid is not mapped to cell
    cell_parquet_ids = cell_array[nuc_selected_idx[0],
                                  nuc_selected_idx[1],
                                  first_cell[nuc_selected_idx]][:,4].astype('int64')

    # Number of cells per nucleoid
    # It will be zero if nucleoid is not mapped to cell
    cells_per_nuc = n_cells_per_nuc_collapsed[nuc_selected_idx]

    # Indices of cells (in the cell array) mapped to each nucleoid. These are not the Segment IDs
    cell_index_nuc = first_cell[nuc_selected_idx]

    # ONLY NO MAPPED NUCLEOIDS: Make repeated ones -1
    # Make cell parquet index "-1"
    cell_parquet_ids[cell_index_nuc == -1] = -1
    
    return cell_index_nuc, cells_per_nuc, nuc_parquet_ids, cell_parquet_ids
    
def reconcile_trenches(ref_df,masked_df,key="Trenchid Timepoint Index"):
    trench_id_tpt_indices_1 = set(ref_df[key].unique().tolist())
    trench_id_tpt_indices_2 = set(masked_df[key].unique().tolist())
    overlap = sorted(list(trench_id_tpt_indices_1&trench_id_tpt_indices_2))
    masked_df_mask = masked_df[key].isin(overlap)
    masked_df = masked_df[masked_df_mask]
    return masked_df



def assign_nuc_stats_to_cells(nuc_df):#, cell_df):
    
    grouped_nuc_df = nuc_df.groupby('Cell Parquet Index', sort = False)
    # grouped_nuc_df = nuc_df.groupby(level='Cell Parquet Index', sort=False)

    # Number of nucleoids
    nuc_number = grouped_nuc_df.apply(len)
    nuc_number.name = 'Number of Nucleoids'
    
    # Create summary DataFrame
    nuc_data_per_cell = pd.DataFrame(nuc_number)

    # Total nucleoid area
    nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df['area'].sum())
    nuc_data_per_cell.rename(columns={'area': 'Total Nucleoid Area'}, inplace=True)

    # Nucleoid separations
    nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df['centroid_y'].apply(np.diff))
    nuc_data_per_cell.rename(columns={'centroid_y': 'Nucleoid Separations'}, inplace=True)
    # Get separation stats
    nuc_data_per_cell['Nucleoid Separation Mean'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.mean)
    nuc_data_per_cell['Nucleoid Separation CV'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.std)/nuc_data_per_cell['Nucleoid Separation Mean']
    nuc_data_per_cell['Nucleoid Separation CV'] = nuc_data_per_cell['Nucleoid Separation CV'].replace({0:np.nan})
    nuc_data_per_cell['Nucleoid Separation Min'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.min, args=(None, None, False, np.inf))
    nuc_data_per_cell['Nucleoid Separation Min'] = nuc_data_per_cell['Nucleoid Separation Min'].replace({np.inf:np.nan})
    nuc_data_per_cell['Nucleoid Separation Max'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.max, args=(None, None, False, -np.inf))
    nuc_data_per_cell['Nucleoid Separation Max'] = nuc_data_per_cell['Nucleoid Separation Max'].replace({-np.inf:np.nan})
    nuc_data_per_cell.drop('Nucleoid Separations', axis=1, inplace=True) # Drop nucleoid separations

    # Get rest of stats
    mean_and_cv_properties = ['area', 'solidity', 'mCherry mean_intensity']
    mean_and_cv_properties_names = ['Nucleoid Area', 'Nucleoid Solidity', 'Nucleoid Mean Intensity']

    mean_only_properties = ['axis_major_length', 'axis_minor_length']
    mean_only_properties_names = ['Nucleoid Major Axis Length', 'Nucleoid Minor Axis Length']

    for i, metric in enumerate(mean_and_cv_properties):
        # Get means
        nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df[metric].mean())
        nuc_data_per_cell.rename(columns={metric: mean_and_cv_properties_names[i] + ' Mean'}, inplace=True)

        # Get CVs
        cv_df = grouped_nuc_df[metric].std()/nuc_data_per_cell[mean_and_cv_properties_names[i] + ' Mean']
        cv_df.name = mean_and_cv_properties_names[i] + ' CV'
        nuc_data_per_cell =  nuc_data_per_cell.join(cv_df)
        # nuc_data_per_cell.rename(columns={metric: mean_and_cv_properties_names[i] + ' CV'}, inplace=True)

    for i, metric in enumerate(mean_only_properties):
        nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df[metric].mean())
        nuc_data_per_cell.rename(columns={metric: mean_only_properties_names[i] + ' Mean'}, inplace=True)

    # nuc_data_per_cell.drop(-1, inplace=True) # Drop unassigned nucleoids
    
    # Merge with cell df
    return nuc_data_per_cell
    
#     cell_df_merged = cell_df.join(nuc_data_per_cell, how='left')
    
#     # Fill NaNs
#     cell_df_merged['Total Nucleoid Area'] = cell_df_merged['Total Nucleoid Area'].fillna(0)
#     cell_df_merged['Number of Nucleoids'] = cell_df_merged['Number of Nucleoids'].fillna(0)
#     cell_df_merged.loc[cell_df_merged['Number of Nucleoids'] == 0, 'Nucleoid Area Mean'] = 0
#     cell_df_merged.loc[cell_df_merged['Number of Nucleoids'] == 0, 'Nucleoid Mean Intensity Mean'] = 0
#     cell_df_merged.loc[cell_df_merged['Number of Nucleoids'] == 0, 'Nucleoid Major Axis Length Mean'] = 0
#     cell_df_merged.loc[cell_df_merged['Number of Nucleoids'] == 0, 'Nucleoid Minor Axis Length Mean'] = 0
#     # Cast number of nucleoids to int
#     cell_df_merged = cell_df_merged.astype({"Number of Nucleoids":int})
    
    # return cell_df_merged # Join to cell Dataframe and return


In [ ]:
yfp_lineages = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/output/",engine='pyarrow',calculate_divisions=True)
mch_regprops = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/analysis/",engine='pyarrow',calculate_divisions=True)

In [ ]:
file_indices_list = sorted(yfp_lineages["File Index"].unique().compute().tolist())
file_indices_list = file_indices_list + [file_indices_list[-1] + 1]

yfp_lineages = yfp_lineages.reset_index().set_index("File Index",sorted=True).repartition(divisions=file_indices_list,force=True)\
                .reset_index().set_index("File Parquet Index",sorted=True).persist()
mch_regprops = mch_regprops.reset_index().set_index("File Index",sorted=True).repartition(divisions=file_indices_list,force=True)\
                .reset_index().set_index("File Parquet Index",sorted=True).persist()

wait(yfp_lineages);
wait(mch_regprops);

mch_regprops= dd.map_partitions(reconcile_trenches,yfp_lineages,mch_regprops,transform_divisions=False,align_dataframes=False).persist()
yfp_lineages= dd.map_partitions(reconcile_trenches,mch_regprops,yfp_lineages,transform_divisions=False,align_dataframes=False).persist()
wait(mch_regprops);
wait(yfp_lineages);

In [ ]:
# Get meta df
empty_output = mch_regprops.get_partition(0).compute().loc[-1:-1]
empty_output["Segment Index"] = []
empty_output["Segment Index"] = empty_output["Segment Index"].astype(int)

empty_output["Cells Per Nucleoid"] = []
empty_output["Cells Per Nucleoid"] = empty_output["Cells Per Nucleoid"].astype(int)

empty_output["Nuc Parquet Index"] = []
empty_output["Nuc Parquet Index"] = empty_output["Nuc Parquet Index"].astype('int64')

empty_output["Cell Parquet Index"] = []
empty_output["Cell Parquet Index"] = empty_output["Cell Parquet Index"].astype('int64')

# Assign nucleoids to cells
mch_regprops = dd.map_partitions(assign_nucleoids_to_cells_df,mch_regprops,yfp_lineages,transform_divisions=False,align_dataframes=False,meta=empty_output).persist()
wait(mch_regprops);

## Get Area, number of nucleoids per cell, and merge with cell df

## 2023-01-20: UPDATE - Let `nuc_data_per_cell` be a Dask Dataframe instead of a pandas Dataframe

In [ ]:
# Generate meta df
empty_df = yfp_lineages.get_partition(0).compute().iloc[-1:-1]
cols_to_remove = empty_df.columns
empty_df = empty_df.reindex(columns=list(empty_df.columns)+
                                   ['Number of Nucleoids', 'Total Nucleoid Area',
       'Nucleoid Separation Mean', 'Nucleoid Separation CV',
       'Nucleoid Separation Min', 'Nucleoid Separation Max',
       'Nucleoid Area Mean', 'Nucleoid Area CV', 'Nucleoid Solidity Mean',
       'Nucleoid Solidity CV', 'Nucleoid Mean Intensity Mean',
       'Nucleoid Mean Intensity CV', 'Nucleoid Major Axis Length Mean',
       'Nucleoid Minor Axis Length Mean'])
empty_df=empty_df.drop(columns=cols_to_remove)
empty_df['Number of Nucleoids'] = empty_df['Number of Nucleoids'].astype('int64')   

# Generate stats df
nuc_data_per_cell_dd = dd.map_partitions(assign_nuc_stats_to_cells,mch_regprops,transform_divisions=False,align_dataframes=False,meta=empty_df).persist()
wait(nuc_data_per_cell_dd);

In [ ]:
# Save mcherry df
dd.to_parquet(mch_regprops, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/analysisNucsMappedToCells", engine="pyarrow", overwrite=True)
del mch_regprops

In [ ]:
# Load again
yfp_lineages = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/output/",engine='pyarrow',calculate_divisions=True)

# Join to stats df
yfp_lineages = yfp_lineages.join(nuc_data_per_cell_dd, how = 'left').persist()
yfp_lineages['Total Nucleoid Area'] = yfp_lineages['Total Nucleoid Area'].fillna(0)
yfp_lineages['Number of Nucleoids'] = yfp_lineages['Number of Nucleoids'].fillna(0)
yfp_lineages['Nucleoid Area Mean'] = yfp_lineages['Nucleoid Area Mean'].fillna(0)
yfp_lineages['Nucleoid Mean Intensity Mean'] = yfp_lineages['Nucleoid Mean Intensity Mean'].fillna(0)
yfp_lineages['Nucleoid Major Axis Length Mean'] = yfp_lineages['Nucleoid Major Axis Length Mean'].fillna(0)
yfp_lineages['Nucleoid Minor Axis Length Mean'] = yfp_lineages['Nucleoid Minor Axis Length Mean'].fillna(0).persist()
wait(yfp_lineages);
# Save
dd.to_parquet(yfp_lineages, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/withNucleoidData", engine="pyarrow", overwrite=True)

In [ ]:
yfp_lineages_2 = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/withNucleoidData/",engine='pyarrow',calculate_divisions=True).persist()

In [ ]:
wait(yfp_lineages_2);

In [ ]:
# empty_cell_df = yfp_lineages.get_partition(0).compute().iloc[-1:-1]
# extra_columns = ['Number of Nucleoids',
#        'Total Nucleoid Area', 'Nucleoid Separation Mean',
#        'Nucleoid Separation CV', 'Nucleoid Separation Min',
#        'Nucleoid Separation Max', 'Nucleoid Area Mean', 'Nucleoid Area CV',
#        'Nucleoid Solidity Mean', 'Nucleoid Solidity CV',
#        'Nucleoid Mean Intensity Mean', 'Nucleoid Mean Intensity CV',
#        'Nucleoid Major Axis Length Mean', 'Nucleoid Minor Axis Length Mean']

# empty_cell_df = empty_cell_df.reindex(columns = list(empty_cell_df.columns) + extra_columns)
# empty_cell_df['Number of Nucleoids'] = empty_cell_df['Number of Nucleoids'].astype('int64')

In [ ]:
# yfp_lineages = dd.map_partitions(assign_nuc_stats_to_cells,mch_regprops,transform_divisions=False,align_dataframes=False,meta=empty_cell_df).persist()
# wait(mch_regprops);

In [ ]:
# grouped_nuc_df = nuc_df.groupby(level='Cell Parquet Index', sort=False)

# # Number of nucleoids
# nuc_number = grouped_nuc_df.apply(len)
# nuc_number.name = 'Number of Nucleoids'

# # Create summary DataFrame
# nuc_data_per_cell = pd.DataFrame(nuc_number)

# # Total nucleoid area
# nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df['area'].sum())
# nuc_data_per_cell.rename(columns={'area': 'Total Nucleoid Area'}, inplace=True)

# # Nucleoid separations
# nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df['centroid_y'].apply(np.diff))
# nuc_data_per_cell.rename(columns={'centroid_y': 'Nucleoid Separations'}, inplace=True)
# # Get separation stats
# nuc_data_per_cell['Nucleoid Separation Mean'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.mean)
# nuc_data_per_cell['Nucleoid Separation CV'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.std)/nuc_data_per_cell['Nucleoid Separation Mean']
# nuc_data_per_cell['Nucleoid Separation CV'] = nuc_data_per_cell['Nucleoid Separation CV'].replace({0:np.nan})
# nuc_data_per_cell['Nucleoid Separation Min'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.min, args=(None, None, False, np.inf))
# nuc_data_per_cell['Nucleoid Separation Min'] = nuc_data_per_cell['Nucleoid Separation Min'].replace({np.inf:np.nan})
# nuc_data_per_cell['Nucleoid Separation Max'] = nuc_data_per_cell['Nucleoid Separations'].apply(np.max, args=(None, None, False, -np.inf))
# nuc_data_per_cell['Nucleoid Separation Max'] = nuc_data_per_cell['Nucleoid Separation Max'].replace({-np.inf:np.nan})
# nuc_data_per_cell.drop('Nucleoid Separations', axis=1, inplace=True) # Drop nucleoid separations

# # Get rest of stats
# mean_and_cv_properties = ['area', 'solidity', 'mCherry mean_intensity']
# mean_and_cv_properties_names = ['Nucleoid Area', 'Solidity', 'mCherry Mean Intensity']

# mean_only_properties = ['axis_major_length', 'axis_minor_length']
# mean_only_properties_names = ['Nucleoid Major Axis Length', 'Nucleoid Minor Axis Length']

# for i, metric in enumerate(mean_and_cv_properties):
#     # Get means
#     nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df[metric].mean())
#     nuc_data_per_cell.rename(columns={metric: mean_and_cv_properties_names[i] + ' Mean'}, inplace=True)
    
#     # Get CVs
#     cv_df = grouped_nuc_df[metric].std()/nuc_data_per_cell[mean_and_cv_properties_names[i] + ' Mean']
#     cv_df.name = mean_and_cv_properties_names[i] + ' CV'
#     nuc_data_per_cell =  nuc_data_per_cell.join(cv_df)
#     # nuc_data_per_cell.rename(columns={metric: mean_and_cv_properties_names[i] + ' CV'}, inplace=True)
    
# for i, metric in enumerate(mean_only_properties):
#     nuc_data_per_cell = nuc_data_per_cell.join(grouped_nuc_df[metric].mean())
#     nuc_data_per_cell.rename(columns={metric: mean_only_properties_names[i] + ' Mean'}, inplace=True)

# nuc_data_per_cell.drop(-1, inplace=True) # Drop unassigned nucleoids

This does it for one variable

Now for two

In [ ]:
# # nuc_df = nuc_df.reset_index().set_index('Cell Parquet Index')
# # nuc_number = nuc_df.index.value_counts(sort = False)
# # nuc_number = nuc_number.rename_axis('Cell Parquet Index')
# # nuc_number.name = 'Number of Nucleoids'

# # nuc_data_per_cell = pd.DataFrame(nuc_number)

# # nuc_area = nuc_df['area'].groupby(level=0, sort=False).sum()
# # nuc_area.name = 'Total Nucleoid Area'
# # nuc_data_per_cell = nuc_data_per_cell.join(nuc_area)

# # nuc_separations = nuc_df.reset_index().groupby('Cell Parquet Index', sort=False).apply(lambda x:np.diff(x['centroid_y'].sort_values()))
# # nuc_separations.name = 'Nucleoid Separations'
# # nuc_data_per_cell = nuc_data_per_cell.join(nuc_separations)

# nuc_separation_cv = nuc_separation_cv.replace({0:np.nan})


## 2023-01-19: UPDATE - Get also nucleoid distance metrics, nucleaoid mean area, and nucleoid mean CV

Write a function to get all the desired mean properties

In [ ]:
mch_regprops = mch_regprops.set_index('Cell Parquet Index').persist()
wait(mch_regprops);

In [ ]:
# Compute number of nucleoids per cell
nuc_number = mch_regprops.index.value_counts(sort = False).compute()
nuc_number.index.name = 'Cell Parquet Index'
nuc_number.name = 'Number of Nucleoids'

When running the previous cell, there is a long delay at the last value_count step

In [ ]:
# Compute total nucleoid area per cell
nuc_total_area = mch_regprops['area'].groupby('Cell Parquet Index').sum().compute()
nuc_total_area.name = 'Total Nucleoid Area'

I noticed a worker with 2x the memory usage of the rest building up, it released in the end

In [ ]:
# Create summary statistics dataframe
nuc_data_per_cell = pd.concat((nuc_number, nuc_total_area), axis=1)
nuc_data_per_cell = nuc_data_per_cell.drop(-1) # Drop unassigned

del nuc_number, nuc_total_area

This is quite slow. It is done locally

In [ ]:
# Add nucleoid distance metrics
nuc_separation_mean = mch_regprops['centroid_y'].groupby('Cell Parquet Index').apply(lambda x : np.mean(np.diff(x.sort_values()))).compute()
nuc_separation_mean.name = 'Nucleoid Separation Mean'

The `lambda` function in dask runs very slowly. Can I do this more efficiently? Memory usage here is good, though

In [ ]:
nuc_separation_std = mch_regprops['centroid_y'].groupby('Cell Parquet Index').apply(lambda x : np.std(np.diff(x.sort_values()))).compute()
nuc_separation_cv = nuc_separation_std/nuc_separation_mean
del nuc_separation_std

This one is also very slow. Adding saving checkpoint here

In [ ]:
nuc_separation_cv = nuc_separation_cv.replace({0:np.nan})
nuc_separation_cv.name = 'Nucleoid Separation CV'

Fast

In [ ]:
nuc_data_per_cell['Nucleoid Separation Mean'] = nuc_separation_mean

This is very slow. Can I do it in a single step? So that it does not copy the DataFrame twice?

In [ ]:
nuc_data_per_cell['Nucleoid Separation CV'] = nuc_separation_cv
del nuc_separation_mean, nuc_separation_cv

In [ ]:
!pwd

In [ ]:
nuc_data_per_cell.to_hdf('/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/nuc_data_per_cell.hdf', key='data')

The file is 2.6 GB large up to here

In [ ]:
# Add other stuff
mean_and_cv_properties = ['area', 'solidity', 'mCherry mean_intensity']
mean_and_cv_properties_names = ['Nucleoid Area', 'Solidity', 'mCherry Mean Intensity']

mean_only_properties = ['axis_major_length', 'axis_minor_length']
mean_only_properties_names = ['Nucleoid Major Axis Length', 'Nucleoid Minor Axis Length']

for i, metric in enumerate(mean_and_cv_properties):
    nuc_metric_mean = mch_regprops[metric].groupby('Cell Parquet Index').mean().compute()
    nuc_metric_mean_name = mean_and_cv_properties_names[i] + ' Mean'
    nuc_metric_mean.name = nuc_metric_mean_name
    
    nuc_metric_std = mch_regprops[metric].groupby('Cell Parquet Index').std().compute()
    nuc_metric_cv  = nuc_metric_std/nuc_metric_mean
    del nuc_metric_std
    nuc_metric_cv_name = mean_and_cv_properties_names[i] + ' CV'
    nuc_metric_cv.name = nuc_metric_cv_name
    
    nuc_data_per_cell[nuc_metric_mean_name] = nuc_metric_mean
    nuc_data_per_cell[nuc_metric_cv_name] = nuc_metric_cv
    del nuc_metric_mean, nuc_metric_cv
    print(i, metric, 'Done')

In [ ]:
nuc_data_per_cell.to_hdf('/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/nuc_data_per_cell.hdf', key='data')

5.7 GB here

In [ ]:
for i, metric in enumerate(mean_only_properties):
    nuc_metric_mean = mch_regprops[metric].groupby('Cell Parquet Index').mean().compute()
    nuc_metric_mean_name = mean_only_properties_names[i] + ' Mean'
    nuc_metric_mean.name = nuc_metric_mean_name
    
    nuc_data_per_cell[nuc_metric_mean_name] = nuc_metric_mean
    del nuc_metric_mean

In [ ]:
nuc_data_per_cell.to_hdf('/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/nuc_data_per_cell.hdf', key='data')

6.8 GB here

In [ ]:
# Read again to keep same partition structure
yfp_lineages = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/output/",engine='pyarrow',calculate_divisions=True).persist()

In [ ]:
yfp_lineages = yfp_lineages.join(nuc_data_per_cell, how = 'left').persist()
wait(yfp_lineages);

The joining is messed up. Maybe since I am trying to merge a normal pandas df with a dask dataframe it cannot partition it so it is trying to do everything on a single worker

In [ ]:
# Turn nans into zeros
yfp_lineages_raw2['Total Nucleoid Area'] = yfp_lineages_raw2['Total Nucleoid Area'].fillna(0).persist()
yfp_lineages_raw2['Number of Nucleoids'] = yfp_lineages_raw2['Number of Nucleoids'].fillna(0).persist()

In [ ]:
dd.to_parquet(yfp_lineages_raw2, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/withNucleoidData", engine="pyarrow", overwrite=True)
dd.to_parquet(mch_regprops, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/analysisNucsMappedToCells", engine="pyarrow", overwrite=True)

In [ ]:
yfp_lineages = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/lineage/withNucleoidData/",engine='pyarrow',calculate_divisions=True)
mch_regprops = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/analysisMappedToCells/",engine='pyarrow',calculate_divisions=True)

In [ ]:
y0 = yfp_lineages.get_partition(0).compute()

In [ ]:
y0.to_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/cells_part0.csv')

In [ ]:
m0 = mch_regprops.get_partition(3).compute()
m0.to_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/nucs_part3.csv')

In [ ]:
pd.read_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/nuc_part0.csv', index_col='File Parquet Index')

In [ ]:
pd.read_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/nuc_part0.csv', index_col='File Parquet Index')

In [ ]:
m0

In [ ]:
m0[m0['File Trench Index'] == 0]

In [ ]:
m0[m0['File Trench Index'] == 0]

## Testing

In [ ]:
mch_regprops[mch_regprops['Cells Per Nucleoid'] > 1].compute()

In [ ]:
mch_regprops[mch_regprops['Cell Parquet Index'] == -1].compute()

In [ ]:
def plot_function(dfy_allt, dfm_allt, timepoint, channel = 'mCherry', use_colors=True, no_yfp = False):
    
    if not no_yfp:
        dfy_single = dfy_allt[dfy_allt["timepoints"] == timepoint]
        dfy_single_indices = dfy_single.index.to_numpy()
    dfm_single = dfm_allt[dfm_allt["timepoints"] == timepoint]
    
    print(dfy_single.columns)
    channels = [channel]
    t_inds = [timepoint]
    with h5py.File("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
        for ch in channels:
            img = infile[ch][idx][t_inds[0]]
            if len(t_inds) > 1:
                for t_ind in t_inds[1:]:
                    img_ind = infile[ch][idx][t_ind]
                    img = np.concatenate((img, img_ind), axis = 1)

    fig, ax = plt.subplots();
    ax.imshow(img, cmap='gray');
    
    if not no_yfp:
        for i in range(len(dfy_single)):
            row = dfy_single.iloc[i]
            # ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')

            n_width = row['bbox_max_col']-row['bbox_min_col']
            n_height = row['bbox_max_row']-row['bbox_min_row']

            # SHOW CELL EDGE WITH COLOR
            ax.plot(row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns, 'o', markersize = 4, color = colors[dfy_single_indices[i]%len(colors)]);
            
            rect = patches.Rectangle((row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns), 
                                     n_width/pixel_microns,
                                     n_height/pixel_microns,
                                     facecolor = 'none', edgecolor='w', linewidth = 1);
            ax.add_patch(rect);
    
    # SHOW NUCLEOID CENTROID
    for i in range(len(dfm_single)):
        if use_colors:
            
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, 
                     color = colors[dfm_single["Cell Parquet Index"].iloc[i]%len(colors)]);
        else:
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, color = 'r');
            

In [ ]:
mch_regprops[mch_regprops['Cells Per Nucleoid'] > 1].compute()

In [ ]:
pixel_microns = 0.105951961895293
kymo_length_y = 290*pixel_microns # CHANGE THIS
kymo_width_x = 31*pixel_microns # CHANGE THIS

In [ ]:
# How do I quickly
# file_idx = 5
# idx = 6
# timepoint = 163

file_idx = 2
idx = 40
timepoint = 187

test_nuc = mch_regprops.get_partition(file_idx).compute().reset_index();
test_lineage = yfp_lineages.get_partition(file_idx).compute();

plot_function(dfy_allt=test_lineage[test_lineage['File Trench Index']==idx], 
              dfm_allt=test_nuc[test_nuc["File Trench Index"]==idx], 
              timepoint=timepoint, 
              channel = 'YFP',
              use_colors=True, 
              no_yfp=False);

print(file_idx, idx, timepoint)

In [ ]:
yfp_lineages = yfp_lineages.join(nuc_data_per_cell, how = 'left').persist()
wait(yfp_lineages);

In [ ]:
yfp_lineages['Total Nucleoid Area'] = yfp_lineages['Total Nucleoid Area'].fillna(0).persist()
yfp_lineages['Number of Nucleoids'] = yfp_lineages['Number of Nucleoids'].fillna(0).persist()

In [ ]:
len(yfp_lineages)

In [ ]:
len(yfp_lineages_raw2)

In [ ]:
yfp_lineages.get_partition(0).compute()

In [ ]:
yfp_lineages_raw2.get_partition(0).compute()

In [ ]:
yfp_lineages_raw2[yfp_lineages_raw2["Global CellID"] == 1000480065].compute()

In [ ]:
yfp_lineages[yfp_lineages["Global CellID"] == 1000480065].compute()

In [ ]:
dask_controller.shutdown()

In [ ]:
len(yfp_lineages)

In [ ]:
len(yfp_lineages)

In [ ]:
yfp_lineages = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/lineage/withNucleoidData",engine='pyarrow',calculate_divisions=True).persist()
mch_regprops = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/analysisMappedToCells",engine='pyarrow',calculate_divisions=True).persist()
wait(yfp_lineages);
wait(mch_regprops);

In [ ]:
part_number = 10
test_cell_data = yfp_lineages.get_partition(part_number).compute()
wt_test_cell_data = test_cell_data[test_cell_data['timepoints'] < 40]

In [ ]:
test_cell_data[test

In [ ]:
plt.figure(figsize = (5,5))

plt.plot(test_cell_data['area'], test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')
# plt.xlim([3,7])
# plt.ylim([0,4])

In [ ]:
plt.figure(figsize = (5,5))

plt.plot(test_cell_data['area'], test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')
plt.xlim([0,10])
plt.ylim([0,4])

In [ ]:
ydf = yfp_lineages.get_partition(2)

In [ ]:
ydf

In [ ]:
ydf[(ydf['area'] > 5) & (ydf['Total Nucleoid Area'] < 1.5)].compute()

In [ ]:
plt.figure(figsize = (5,5))

plt.plot(test_cell_data['area'], test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
plt.figure(figsize = (5,5))

plt.plot(test_cell_data['area'], test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
wt_t0 = yfp_lineages.groupby(['File Index', 'File Trench Index']).first().compute()

In [ ]:
wt_t0['timepoints'].unique()

In [ ]:
cr_t200 = yfp_lineages.groupby(['File Index', 'File Trench Index']).last().compute()

In [ ]:
wt_t0

In [ ]:
wt_t200['Number of Nucleoids'].astype('int64')

In [ ]:
import seaborn as sns

In [ ]:
wt_t200['Number of Nucleoids']

In [ ]:
a = cr_t200[cr_t200['Number of Nucleoids'] == 1]

In [ ]:
a

In [ ]:
sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = cr_t200, alpha = 0.5)
sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = a, alpha = 0.5)
sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', data = wt_t0, alpha = 0.5)

In [ ]:
cr_t200[(cr_t200['Total Nucleoid Area'] > 7) & (cr_t200['Total Nucleoid Area'] < 13) & (cr_t200['area'] > 23) & (cr_t200['area'] < 27) & (cr_t200['Number of Nucleoids'] == 1)]

In [ ]:
cr_t200[(cr_t200['Total Nucleoid Area'] > 7) & (cr_t200['Total Nucleoid Area'] < 13) & (cr_t200['area'] > 23) & (cr_t200['area'] < 27)]

In [ ]:
a[(a['Total Nucleoid Area'] < 5) & (a['area'] > 15)]

In [ ]:
t_lo = 0
t_hi = 40

def aggregate_per_trench_mean(dd, column, t_lo = 0, t_hi = 40):
    return dd[(dd['timepoints'] >= t_lo) & (dd['timepoints'] <= t_hi)].groupby(['File Index', 'File Trench Index'])[column].mean().compute()
# mean_area_wt = yfp_lineages[(yfp_lineages['timepoints'] >= t_lo) & (yfp_lineages['timepoints'] <= t_hi)].groupby(['File Index', 'File Trench Index'])['area'].mean().compute()


In [ ]:
def aggregate_per_trench_mode(dd, column, t_lo = 0, t_hi = 40):
    return dd[(dd['timepoints'] >= t_lo) & (dd['timepoints'] <= t_hi)].groupby(['File Index', 'File Trench Index'])[column]

In [ ]:
med_area_wt = aggregate_per_trench_mean(yfp_lineages, 'area', 0, 40)
med_nucarea_wt = aggregate_per_trench_mean(yfp_lineages, 'Total Nucleoid Area', 0, 40)
med_nnuc_wt = aggregate_per_trench_mean(yfp_lineages, 'Number of Nucleoids', 0, 40)

In [ ]:
mod_nnuc_wt = aggregate_per_trench_mode(yfp_lineages, 'Number of Nucleoids', 0, 40)

In [ ]:
med_area_f = aggregate_per_trench_mean(yfp_lineages, 'area', 160, 200)
med_nucarea_f = aggregate_per_trench_mean(yfp_lineages, 'Total Nucleoid Area', 160, 200)
med_nnuc_f = aggregate_per_trench_mean(yfp_lineages, 'Number of Nucleoids', 160, 200)

In [ ]:
med_all_f = pd.concat((med_area_f, med_nucarea_f, med_nnuc_f), axis=1)

In [ ]:
mod_nnuc_wt

In [ ]:
med_all_wt = pd.concat((med_area_wt, med_nucarea_wt, med_nnuc_wt), axis=1)

In [ ]:
med_all_wt = med_all_wt[med_all_wt['Number of Nucleoids'] <= 3]

In [ ]:
med_all_weird = med_all_wt[(med_all_wt['area'] < 5) & (med_all_wt['Total Nucleoid Area'] < 0.3)]
med_all_weird

In [ ]:
import seaborn as sns

In [ ]:
# Number of nucleoid (mode per group)

In [ ]:
plt.figure(figsize = (5,5))

sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = med_all_wt, alpha = 0.5)
# sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = med_all_wt, alpha = 0.5)

plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
plt.figure(figsize = (5,5))

sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = med_all_f, alpha = 0.5)
# sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = med_all_wt, alpha = 0.5)

plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
plt.figure(figsize = (7,7))

sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', data = med_all_f, alpha = 0.2, label = '10h after CRISPRi induction')
sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', data = med_all_wt, alpha = 0.2, label = 'Before CRISPRi induction')
# sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', data = med_all_f, alpha = 0.5)
# sns.scatterplot(x = 'area', y = 'Total Nucleoid Area', hue = 'Number of Nucleoids', data = med_all_wt, alpha = 0.5)

plt.xlabel('Cell Area ($\mu m^2$)', fontsize = 20)
plt.ylabel('Total Nucleoid Area ($\mu m^2$)', fontsize = 20)

plt.legend(fontsize = 15)

_ = plt.xticks(fontsize=15)
_ = plt.yticks(fontsize=15)

In [ ]:
area_lo = 20
area_hi = 25

n_area_lo = 7
n_area_hi = 11
med_all_f[(med_all_f['area'] > area_lo) & (med_all_f['area'] < area_hi) & (med_all_f['Total Nucleoid Area'] > n_area_lo) & (med_all_f['Total Nucleoid Area'] < n_area_hi)]#.groupby(['File Index', 'File Trench Index', 'timepoints']).first().compute()

In [ ]:
# area_lo = 20
# area_hi = 25

# n_area_lo = 0
# n_area_hi = 4
# yfp_lineages[(yfp_lineages['area'] > area_lo) & (yfp_lineages['area'] < area_hi) & (yfp_lineages['Total Nucleoid Area'] > n_area_lo) & (yfp_lineages['Total Nucleoid Area'] < n_area_hi)].groupby(['File Index', 'File Trench Index', 'timepoints']).first().compute()

In [ ]:
# How do I quickly
# file_idx = 5
# idx = 6
# timepoint = 163

file_idx = 0
idx = 1
timepoint = 9

test_nuc = mch_regprops.get_partition(file_idx).compute().reset_index();
test_lineage = yfp_lineages.get_partition(file_idx).compute();

plot_function(dfy_allt=test_lineage[test_lineage['File Trench Index']==idx], 
              dfm_allt=test_nuc[test_nuc["File Trench Index"]==idx], 
              timepoint=timepoint, 
              channel = 'YFP',
              use_colors=True, 
              no_yfp=False);

print(file_idx, idx, timepoint)

In [ ]:
plt.figure(figsize = (5,5))


plt.plot(wt_t200['area'], wt_t200['Total Nucleoid Area'], 'o', alpha = 0.05, c=)
plt.plot(wt_t0['area'], wt_t0['Total Nucleoid Area'], 'o', alpha = 0.05)
# plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
c_area = wt_test_cell_data.groupby('File Trench Index')['area'].median()
n_area = wt_test_cell_data.groupby('File Trench Index')['Total Nucleoid Area'].median()

In [ ]:
plt.figure(figsize = (5,5))

plt.plot(wt_test_cell_data['area'], wt_test_cell_data['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(c_area, n_area, 'o', alpha = 0.3)
plt.plot(l_c_area, l_n_area, 'o', alpha = 0.5)
plt.xlabel('Cell Area')
plt.ylabel('Nucleoid Area')

In [ ]:
200 imaged every 3 minutes, 600 minutes

In [ ]:
last_cell_data = test_cell_data[test_cell_data['timepoints'] > 180]
l_c_area = last_cell_data.groupby('File Trench Index')['area'].median()
l_n_area = last_cell_data.groupby('File Trench Index')['Total Nucleoid Area'].median()

In [ ]:
c_area = wt_test_cell_data.groupby('File Trench Index')['area'].median()
n_area = wt_test_cell_data.groupby('File Trench Index')['Total Nucleoid Area'].median()

In [ ]:
last_cell_data[(last_cell_data['area'] > 7.5) & (last_cell_data['Total Nucleoid Area'] < 250)]['File Trench Index'].unique()

In [ ]:
def plot_function(dfy_allt, dfm_allt, timepoint, channel = 'mCherry', use_colors=True, no_yfp = False):
    
    if not no_yfp:
        dfy_single = dfy_allt[dfy_allt["timepoints"] == timepoint]
        dfy_single_indices = dfy_single.index.to_numpy()
    dfm_single = dfm_allt[dfm_allt["timepoints"] == timepoint]
    
    print(dfy_single.columns)
    channels = [channel]
    t_inds = [timepoint]
    with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
        for ch in channels:
            img = infile[ch][idx][t_inds[0]]
            if len(t_inds) > 1:
                for t_ind in t_inds[1:]:
                    img_ind = infile[ch][idx][t_ind]
                    img = np.concatenate((img, img_ind), axis = 1)

    fig, ax = plt.subplots();
    ax.imshow(img, cmap='gray');
    
    if not no_yfp:
        for i in range(len(dfy_single)):
            row = dfy_single.iloc[i]
            # ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')

            n_width = row['bbox_max_col']-row['bbox_min_col']
            n_height = row['bbox_max_row']-row['bbox_min_row']

            # SHOW CELL EDGE WITH COLOR
            ax.plot(row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns, 'o', markersize = 4, color = colors[dfy_single_indices[i]%len(colors)]);
            
            rect = patches.Rectangle((row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns), 
                                     n_width/pixel_microns,
                                     n_height/pixel_microns,
                                     facecolor = 'none', edgecolor='w', linewidth = 1);
            ax.add_patch(rect);
    
    # SHOW NUCLEOID CENTROID
    for i in range(len(dfm_single)):
        if use_colors:
            
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, 
                     color = colors[dfm_single["Cell Parquet Index"].iloc[i]%len(colors)]);
        else:
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, color = 'r');
            

In [ ]:
# How do I quickly
# file_idx = 5
# idx = 6
# timepoint = 163 

file_idx = 500
idx = 37
timepoint = 80

test_nuc = mch_regprops.get_partition(file_idx).compute().reset_index();
test_lineage = yfp_lineages.get_partition(file_idx).compute();

# plot_function(dfy_allt=test_lineage, dfm_allt=test_nuc, timepoint=0, use_colors=False)

In [ ]:
plot_function(dfy_allt=test_lineage[test_lineage['File Trench Index']==idx], 
              dfm_allt=test_nuc[test_nuc["File Trench Index"]==idx], 
              timepoint=timepoint, 
              channel = 'mCherry',
              use_colors=True, 
              no_yfp=False);

In [ ]:
# Save dataframe
#
# dd.to_parquet(kymo_df, "/home/de64/scratch/de64/sync_folder/2022-01-20_lDE20_Final_6/Growth_Division/kymograph/metadata_timefix", engine="pyarrow", overwrite=True)